# Tutorial 07: Synthetic Data Generation

Learn how to generate synthetic VRP instances for testing, benchmarking, and algorithm development.

**What you'll learn:**
- Use OrderGenerator for random instance creation
- Generate instances with specific characteristics
- Create benchmark datasets
- Control problem difficulty and properties
- Generate test suites for algorithm validation

**Prerequisites:**
- Tutorial 01 (Quickstart)
- Tutorial 03 (Custom Problems)

**Time:** ~30 minutes

## 1. Setup and Imports

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

# VRP Toolkit imports
from vrp_toolkit.data.generators import OrderGenerator, DemandGenerator
from vrp_toolkit.problems.pdptw import PDPTWInstance
from vrp_toolkit.algorithms.alns.solver import ALNSSolver

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

print("All imports successful!")

## 2. Quick Start: Generate Random Instance

Let's create the **simplest possible random instance** with OrderGenerator.

In [ ]:
# Create generator
generator = OrderGenerator(num_orders=5, num_vehicles=2)

# Generate instance
instance = generator.generate_instance()

print(f"Generated instance with:")
print(f"  Orders: {instance.n}")
print(f"  Total nodes: {len(instance.nodes)}")
print(f"  Vehicle capacity: {instance.vehicle_capacity}")
print(f"  Battery capacity: {instance.battery_capacity}")

# Solve it
solver = ALNSSolver()
solution = solver.solve(instance)
print(f"\nSolution cost: {solution.objective_value():.2f}")

**What just happened:**
- OrderGenerator created random pickup/delivery locations
- Assigned random demands and time windows
- Built complete PDPTWInstance with distance matrix
- Ready to solve immediately!

## 3. Understanding OrderGenerator

### 3.1 Basic Parameters

**OrderGenerator** accepts these key parameters:
```python
OrderGenerator(
    num_orders=10,          # Number of pickup-delivery pairs
    num_vehicles=3,         # Number of vehicles
    grid_size=100,          # Coordinate range [0, grid_size)
    demand_range=(1, 10),   # Random demand range
    time_horizon=480,       # Total time available (minutes)
    seed=None               # Random seed for reproducibility
)
```

In [ ]:
# Example: Customized generation
custom_gen = OrderGenerator(
    num_orders=8,
    num_vehicles=2,
    grid_size=50,          # Smaller area
    demand_range=(2, 8),   # Moderate demands
    time_horizon=300,      # 5-hour shift
    seed=123               # Reproducible
)

custom_instance = custom_gen.generate_instance()

print("Custom instance properties:")
print(f"  Grid size: {custom_gen.grid_size}")
print(f"  Demand range: {custom_gen.demand_range}")
print(f"  Time horizon: {custom_gen.time_horizon}")
print(f"  Max route time: {custom_instance.max_route_time}")

### 3.2 Controlling Problem Characteristics

Different parameter combinations create different problem types:

In [ ]:
# Easy problem: Few orders, loose constraints
easy_gen = OrderGenerator(
    num_orders=3,
    num_vehicles=2,
    grid_size=50,
    time_horizon=1000  # Very relaxed
)
easy_instance = easy_gen.generate_instance()

# Hard problem: Many orders, tight constraints
hard_gen = OrderGenerator(
    num_orders=20,
    num_vehicles=3,
    grid_size=100,
    time_horizon=200   # Tight
)
hard_instance = hard_gen.generate_instance()

# Solve both
easy_solution = solver.solve(easy_instance)
hard_solution = solver.solve(hard_instance)

print("Easy problem:")
print(f"  Orders: {easy_instance.n}, Cost: {easy_solution.objective_value():.2f}")
print(f"\nHard problem:")
print(f"  Orders: {hard_instance.n}, Cost: {hard_solution.objective_value():.2f}")

### 3.3 Reproducibility with Seeds

**When to use seeds:**
- Comparing algorithms on same instance
- Debugging
- Creating benchmark datasets
- Sharing instances with others

In [ ]:
# Same seed = same instance
gen1 = OrderGenerator(num_orders=5, seed=999)
gen2 = OrderGenerator(num_orders=5, seed=999)

instance1 = gen1.generate_instance()
instance2 = gen2.generate_instance()

# Check if identical
print("Are instances identical?")
print(f"  Same number of nodes: {len(instance1.nodes) == len(instance2.nodes)}")
print(f"  Same distance matrix: {np.allclose(instance1.distance_matrix, instance2.distance_matrix)}")
print(f"  Same first node location: {(instance1.nodes[1].x, instance1.nodes[1].y) == (instance2.nodes[1].x, instance2.nodes[1].y)}")

## 4. Advanced Generation Techniques

### 4.1 Clustered Locations

**Use case:** Modeling delivery zones, neighborhoods

In [ ]:
def generate_clustered_instance(num_orders, num_clusters=3):
    """
    Generate instance with spatially clustered orders.
    
    Args:
        num_orders: Total number of orders
        num_clusters: Number of spatial clusters
    """
    # Generate cluster centers
    centers = [(random.uniform(20, 80), random.uniform(20, 80)) for _ in range(num_clusters)]
    
    # Assign orders to clusters
    nodes = [Node(node_id=0, x=50, y=50, node_type='depot')]  # Depot at center
    
    for i in range(1, num_orders + 1):
        # Pick random cluster
        center_x, center_y = random.choice(centers)
        
        # Generate pickup near cluster center
        pickup_x = center_x + random.gauss(0, 10)
        pickup_y = center_y + random.gauss(0, 10)
        
        # Delivery also near same cluster
        delivery_x = center_x + random.gauss(0, 10)
        delivery_y = center_y + random.gauss(0, 10)
        
        pickup_id = i
        delivery_id = i + num_orders
        
        from vrp_toolkit.problems.pdptw import Node
        nodes.append(Node(
            node_id=pickup_id,
            x=pickup_x, y=pickup_y,
            demand=random.uniform(1, 10),
            time_window=(0, 480),
            service_time=5,
            node_type='pickup',
            pair_node_id=delivery_id
        ))
        
        nodes.append(Node(
            node_id=delivery_id,
            x=delivery_x, y=delivery_y,
            demand=-nodes[-1].demand,
            time_window=(0, 480),
            service_time=5,
            node_type='delivery',
            pair_node_id=pickup_id
        ))
    
    return PDPTWInstance(
        nodes=nodes,
        battery_capacity=1000,
        max_route_time=480,
        vehicle_capacity=50
    )

# Generate clustered instance
clustered = generate_clustered_instance(num_orders=12, num_clusters=3)

# Visualize
fig, ax = plt.subplots(figsize=(8, 8))
pickups = [n for n in clustered.nodes if n.node_type == 'pickup']
deliveries = [n for n in clustered.nodes if n.node_type == 'delivery']

ax.scatter([clustered.nodes[0].x], [clustered.nodes[0].y], c='red', s=200, marker='s', label='Depot')
ax.scatter([n.x for n in pickups], [n.y for n in pickups], c='blue', s=100, marker='^', label='Pickup')
ax.scatter([n.x for n in deliveries], [n.y for n in deliveries], c='green', s=100, marker='v', label='Delivery')

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Clustered Instance (3 clusters)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print("Clustered instance: Pickup/delivery pairs are grouped spatially")

### 4.2 Time Window Patterns

**Use case:** Morning vs afternoon deliveries, rush hour

In [ ]:
def generate_time_window_instance(num_orders, pattern='morning_afternoon'):
    """
    Generate instance with specific time window patterns.
    
    Patterns:
        'morning_afternoon': Half in morning (8-12), half afternoon (13-17)
        'tight': 1-hour windows
        'rush_hour': Avoid 12-13 (lunch break)
    """
    generator = OrderGenerator(num_orders=num_orders)
    instance = generator.generate_instance()
    
    # Modify time windows
    for i, node in enumerate(instance.nodes):
        if node.node_type == 'pickup':
            if pattern == 'morning_afternoon':
                if i % 2 == 0:
                    node.time_window = (8*60, 12*60)  # Morning (in minutes)
                else:
                    node.time_window = (13*60, 17*60)  # Afternoon
            
            elif pattern == 'tight':
                start = random.uniform(8*60, 16*60)
                node.time_window = (start, start + 60)  # 1-hour window
            
            elif pattern == 'rush_hour':
                hour = random.choice([8, 9, 10, 11, 14, 15, 16, 17])
                node.time_window = (hour*60, (hour+2)*60)
        
        elif node.node_type == 'delivery':
            # Delivery window = pickup window + 1 hour
            pickup_node = instance.nodes[node.pair_node_id]
            node.time_window = (pickup_node.time_window[0] + 30,
                                pickup_node.time_window[1] + 120)
    
    return instance

# Generate instances with different patterns
patterns = ['morning_afternoon', 'tight', 'rush_hour']

for pattern in patterns:
    inst = generate_time_window_instance(num_orders=6, pattern=pattern)
    sol = solver.solve(inst)
    print(f"{pattern:20s}: Cost = {sol.objective_value():6.2f}, Feasible = {sol.is_feasible()}")

### 4.3 Demand Distribution Control

**Use case:** Modeling heavy vs light packages, bulk orders

In [ ]:
# Different demand distributions
distributions = [
    ('Uniform', lambda: random.uniform(1, 10)),
    ('Heavy-tailed', lambda: random.expovariate(0.2)),
    ('Normal', lambda: max(1, random.gauss(5, 2))),
    ('Heavy items', lambda: random.uniform(8, 15))
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (name, demand_fn) in enumerate(distributions):
    # Generate demands
    demands = [demand_fn() for _ in range(20)]
    
    # Plot distribution
    axes[idx].hist(demands, bins=15, alpha=0.7, edgecolor='black')
    axes[idx].set_title(f'{name} Distribution')
    axes[idx].set_xlabel('Demand')
    axes[idx].set_ylabel('Frequency')
    axes[idx].axvline(np.mean(demands), color='red', linestyle='--', label=f'Mean: {np.mean(demands):.1f}')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Different demand distributions affect problem difficulty:")
print("- Uniform: Predictable capacity usage")
print("- Heavy-tailed: Few very large orders")
print("- Normal: Most orders near average")
print("- Heavy items: Capacity becomes limiting factor")

## 5. Creating Benchmark Datasets

Generate suites of instances for algorithm testing.

In [ ]:
def create_benchmark_suite(sizes=[5, 10, 20], instances_per_size=5):
    """
    Create a benchmark suite with varying problem sizes.
    
    Args:
        sizes: List of problem sizes (number of orders)
        instances_per_size: How many instances per size
    
    Returns:
        List of (name, instance) tuples
    """
    suite = []
    
    for size in sizes:
        for i in range(instances_per_size):
            # Use different seed for each instance
            generator = OrderGenerator(
                num_orders=size,
                num_vehicles=max(2, size // 5),
                seed=1000 + size*100 + i
            )
            instance = generator.generate_instance()
            
            name = f"instance_{size:02d}_orders_{i+1:02d}"
            suite.append((name, instance))
    
    return suite

# Create benchmark suite
benchmark = create_benchmark_suite(sizes=[5, 10, 15], instances_per_size=3)

print(f"Created benchmark suite with {len(benchmark)} instances:")
for name, inst in benchmark[:6]:  # Show first 6
    print(f"  {name}: {inst.n} orders")
print("  ...")

In [ ]:
# Test algorithm on benchmark suite
results = []

for name, instance in benchmark:
    solution = solver.solve(instance)
    
    results.append({
        'Instance': name,
        'Orders': instance.n,
        'Cost': solution.objective_value(),
        'Routes': len(solution.routes),
        'Feasible': solution.is_feasible()
    })

# Display results
df_results = pd.DataFrame(results)
print("\nBenchmark Results:")
print(df_results.to_string(index=False))

# Summary statistics
print(f"\nSummary:")
print(f"  Average cost: {df_results['Cost'].mean():.2f}")
print(f"  Feasibility rate: {df_results['Feasible'].sum() / len(df_results) * 100:.0f}%")

## 6. Real-World Example: Parameter Sensitivity Analysis

Use generated instances to study how problem parameters affect solution quality.

In [ ]:
# Study: How does number of orders affect solution cost?
order_counts = range(5, 31, 5)
costs_by_size = []

for num_orders in order_counts:
    costs = []
    
    # Generate 10 instances per size
    for seed in range(10):
        gen = OrderGenerator(num_orders=num_orders, seed=2000+seed)
        inst = gen.generate_instance()
        sol = solver.solve(inst)
        costs.append(sol.objective_value())
    
    costs_by_size.append({
        'orders': num_orders,
        'mean_cost': np.mean(costs),
        'std_cost': np.std(costs)
    })

# Visualize relationship
df_sensitivity = pd.DataFrame(costs_by_size)

plt.figure(figsize=(10, 6))
plt.errorbar(df_sensitivity['orders'], df_sensitivity['mean_cost'], 
             yerr=df_sensitivity['std_cost'], 
             marker='o', capsize=5, linewidth=2)
plt.xlabel('Number of Orders')
plt.ylabel('Solution Cost')
plt.title('How Problem Size Affects Solution Cost')
plt.grid(True, alpha=0.3)
plt.show()

print("Key findings:")
print("- Cost increases with problem size (expected)")
print("- Variance also increases (harder to predict)")
print(f"- Cost grows roughly {df_sensitivity['mean_cost'].iloc[-1] / df_sensitivity['mean_cost'].iloc[0]:.1f}x for {order_counts[-1]/order_counts[0]:.1f}x more orders")

## 7. Saving and Loading Generated Instances

**Use case:** Share instances, reproduce experiments

In [ ]:
import pickle

# Generate instance
gen = OrderGenerator(num_orders=10, seed=555)
instance_to_save = gen.generate_instance()

# Save to file
with open('benchmark_instance.pkl', 'wb') as f:
    pickle.dump(instance_to_save, f)

print("Instance saved to benchmark_instance.pkl")

# Load from file
with open('benchmark_instance.pkl', 'rb') as f:
    loaded_instance = pickle.load(f)

print(f"\nLoaded instance: {loaded_instance.n} orders")
print(f"Same as original: {loaded_instance.n == instance_to_save.n}")

# Alternative: Save as CSV
order_table = loaded_instance.order_table
order_table.to_csv('benchmark_instance.csv', index=False)
print("\nAlso saved as CSV for human readability")

## 8. Best Practices

**When generating test data:**

1. **Use seeds for reproducibility:** Always set seeds for benchmark instances
2. **Generate multiple instances:** Don't tune algorithms on single instance
3. **Vary difficulty:** Test on easy, medium, hard problems
4. **Match real-world patterns:** Clustered locations, time window patterns
5. **Document generation params:** Save generator settings with instances

**Common pitfalls:**
- **Too small/large:** 5-20 orders good for testing, 50+ for benchmarking
- **Unrealistic constraints:** Check generated time windows are feasible
- **No variation:** Need diversity in test suite
- **Overfitting:** Don't optimize algorithm on same generated instances repeatedly

**Generation strategies:**
- **Algorithm development:** Small (5-10 orders), many variations
- **Performance testing:** Large (20-50 orders), fewer instances
- **Comparison studies:** Fixed seeds, multiple sizes
- **Stress testing:** Extreme parameters (very tight windows, high demands)

## 9. Practice Exercises

1. **Basic:** Generate 3 instances with identical parameters but different seeds. Verify they produce different solutions.

2. **Intermediate:** Create a "difficulty" function that predicts problem difficulty based on parameters (num_orders, time_horizon, etc.). Test on 20 generated instances.

3. **Advanced:** Generate a benchmark suite mimicking real-world delivery patterns: morning cluster (residential), afternoon cluster (commercial), tight lunch-time windows.

**Hints:**
- For Exercise 1: Use seeds 1, 2, 3
- For Exercise 2: Difficulty might correlate with orders/time_horizon ratio
- For Exercise 3: Combine clustered generation with time window patterns

In [ ]:
# Your solutions here


## 10. Summary

**What you learned:**
- ✅ Generate random PDPTW instances with OrderGenerator
- ✅ Control problem characteristics (size, difficulty, patterns)
- ✅ Create benchmark datasets for algorithm testing
- ✅ Perform parameter sensitivity analysis
- ✅ Save and load generated instances

**Key takeaways:**
1. OrderGenerator creates complete, solvable instances instantly
2. Seeds enable reproducibility for comparisons
3. Varying parameters creates diverse test suites
4. Benchmark suites should span multiple problem sizes and difficulties
5. Generated data complements real-world data for testing

**Next steps:**
- Create your own benchmark suite for your research
- Try **Tutorial 02: Real-World Maps** for realistic distance matrices
- Try **Tutorial 06: Custom Algorithms** and test on generated instances
- Explore standard VRP benchmarks (Solomon, Li & Lim)